# RL climb-agent — 3D long-route training (toprope, 8.7 m target)
Self-driving: clones the mirror repo, installs deps, trains on GPU (~350k steps, warm-started from the 5 m model), then commits the results back to the mirror repo so they can be pulled locally.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
import subprocess, sys, os, time, glob, zipfile
REPO = 'https://github.com/tototofu123/rl-climb-agent-colab.git'
DIR = 'rl-climb-agent'

def sh(cmd, check=True):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (p.stdout[-4000:] + '\n' + p.stderr[-2000:])
    print(tail)
    if check and p.returncode != 0:
        raise RuntimeError(f'failed: {cmd}')
    return p

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
sh(f'git clone --depth 1 {REPO} {DIR}', check=False)
os.chdir(DIR)
sh('git lfs install', check=False)
sh(f'{sys.executable} -m pip install -q gymnasium stable-baselines3 numpy')
print('SETUP DONE')


In [ ]:
!python3 training/train_3d.py --wall toprope --long-route --curriculum --difficulty easy --route-seed 0 --max-slots 24 --max-steps 500 --timesteps 350000 --n-envs 8 --save-freq 25000 --eval-freq 25000 --resume agents/ppo3d_toprope_long_easy_1792944699_final.zip 2>&1 | tail -40
print('TRAIN CELL DONE')

In [ ]:
# package results and push them back to the mirror repo
import subprocess, glob, os, time
files = sorted(glob.glob('agents/ppo3d_toprope_long_easy_*')) + ['experiments/train3d_full_ext.log']
print('result files:', files)
subprocess.run('git config user.email colab@local && git config user.name colab && git add -f agents experiments/train3d_full_ext.log', shell=True, check=False)
subprocess.run('git commit -m "colab training results"', shell=True, check=False)
p = subprocess.run('git push origin HEAD:main', shell=True, capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-2000:])
print('PUSH DONE rc=', p.returncode)